In [7]:
import keras
from keras import optimizers
from keras.callbacks import ModelCheckpoint
from keras.layers import Convolution2D, MaxPooling2D, ZeroPadding2D
from keras.models import Sequential
from keras.preprocessing import image
import matplotlib.pyplot as plt
import numpy as np
import os
from PIL import Image, ImageOps
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras import Model, layers, models
from tensorflow.keras.callbacks import TensorBoard
from tensorflow.keras.layers import (Activation, AvgPool2D, BatchNormalization, Concatenate, 
                                     Conv2D, Dense, Dropout, ELU, Flatten, GlobalAveragePooling2D, 
                                     GlobalAvgPool2D, Input, MaxPool2D, MaxPooling2D, ReLU, 
                                     SeparableConv2D, UpSampling2D)
from tensorflow.keras.models import Model, Sequential, model_from_json
from tensorflow.keras.optimizers import RMSprop, SGD
from tensorflow.keras.preprocessing.image import ImageDataGenerator, array_to_img, img_to_array, load_img
from tensorflow.keras.regularizers import l2
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input, MobileNetV2
import tensorflow_datasets as tfds
from tqdm import tqdm

In [16]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 123

# 1) Load dataset
ds_all, info = tfds.load(
    "plant_village",
    split="train",
    as_supervised=True,   # returns (image, label)
    with_info=True
)

total = info.splits["train"].num_examples
train_size = int(0.8 * total)
val_size = total - train_size

num_classes = info.features["label"].num_classes
print("Number of classes:", num_classes)

# 2) Shuffle then split (reproducible)
ds_all = ds_all.shuffle(
    buffer_size=min(total, 10_000),  # can set total for full shuffle; 10k is faster
    seed=SEED,
    reshuffle_each_iteration=False
)

ds_train_raw = ds_all.take(train_size)
ds_val_raw = ds_all.skip(train_size)

# 3) Augmentation (train only)
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

# 4) Preprocess for MobileNetV2: resize + float + scale to [-1, 1]
def preprocess_mobilenet(image, label):
    image = tf.image.resize(image, IMG_SIZE)  # Resize image
    image = tf.cast(image, tf.float32)    # Ensure float32
    image = preprocess_input(image)  # MobileNetV2 preprocess: expects values in [0,255] and maps to [-1,1]
    return image, label


def preprocess_mobilenet_train(image, label):
    image, label = preprocess_mobilenet(image, label)
    image = data_augmentation(image, training=True)
    return image, label

# 5) Map → Batch → Prefetch


ds_train = (
    ds_train_raw
      .map(preprocess_mobilenet_train, num_parallel_calls=tf.data.AUTOTUNE)
      .batch(BATCH_SIZE)
      .prefetch(tf.data.AUTOTUNE)
)

ds_val = (
    ds_val_raw
      .map(preprocess_mobilenet, num_parallel_calls=tf.data.AUTOTUNE)
      .batch(BATCH_SIZE)
      .prefetch(tf.data.AUTOTUNE)
)

print(f"Total: {total} | Train: {train_size} | Val: {val_size}")
print("Train element_spec:", ds_train.element_spec)

# Optional sanity check: value range should be ~[-1,1]
x_batch, y_batch = next(iter(ds_train))
print("x batch:", x_batch.shape, x_batch.dtype,
      "min:", tf.reduce_min(x_batch).numpy(),
      "max:", tf.reduce_max(x_batch).numpy())
print("y batch:", y_batch.shape, y_batch.dtype)

Number of classes: 38
Total: 54303 | Train: 43442 | Val: 10861
Train element_spec: (TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))
x batch: (32, 224, 224, 3) <dtype: 'float32'> min: -1.0 max: 0.9876281
y batch: (32,) <dtype: 'int64'>


In [13]:
mobile = tf.keras.applications.MobileNetV2()

14536120/14536120 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [14]:
mobile.summary()

Model: "mobilenetv2_1.00_224"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer_2[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 3,538,984 (13.50 MB)

 Trainable params: 3,504,872 (13.37 MB)

 Non-trainable params: 34,112 (133.25 KB)

In [17]:
x = mobile.layers[-6].output
output = Dense(units =38, activation = 'softmax')(x)

In [18]:
model = Model(inputs=mobile.input, outputs=output)

In [20]:
for layer in model.layers[:-23]:
    layer.trainable = False

In [21]:
model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer_2[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 1,855,462 (7.08 MB)

 Trainable params: 961,958 (3.67 MB)

 Non-trainable params: 893,504 (3.41 MB)

In [25]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])

In [26]:
model.fit(x=ds_train, validation_data=ds_val, epochs=30, verbose=2)

Epoch 1/30


ValueError: Arguments `target` and `output` must have the same rank (ndim). Received: target.shape=(None,), output.shape=(None, 7, 7, 38)

In [27]:
base = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)
base.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))
x = base(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)   # <— this removes (7,7)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",  # labels are integers (None,)
    metrics=["accuracy"]
)

model.fit(ds_train, validation_data=ds_val, epochs=30, verbose=2)

Epoch 1/30
1358/1358 - 403s - 297ms/step - accuracy: 0.8567 - loss: 0.5232 - val_accuracy: 0.9216 - val_loss: 0.2538
Epoch 2/30
1358/1358 - 372s - 274ms/step - accuracy: 0.9256 - loss: 0.2394 - val_accuracy: 0.9312 - val_loss: 0.2136
Epoch 3/30
1358/1358 - 563s - 415ms/step - accuracy: 0.9333 - loss: 0.2061 - val_accuracy: 0.9392 - val_loss: 0.1925
Epoch 4/30
1358/1358 - 364s - 268ms/step - accuracy: 0.9397 - loss: 0.1848 - val_accuracy: 0.9410 - val_loss: 0.1898
Epoch 5/30
1358/1358 - 362s - 267ms/step - accuracy: 0.9424 - loss: 0.1770 - val_accuracy: 0.9395 - val_loss: 0.1848
Epoch 6/30
1358/1358 - 373s - 275ms/step - accuracy: 0.9433 - loss: 0.1694 - val_accuracy: 0.9337 - val_loss: 0.2090
Epoch 7/30
1358/1358 - 375s - 276ms/step - accuracy: 0.9453 - loss: 0.1639 - val_accuracy: 0.9447 - val_loss: 0.1779
Epoch 8/30
1358/1358 - 380s - 280ms/step - accuracy: 0.9457 - loss: 0.1606 - val_accuracy: 0.9447 - val_loss: 0.1734
Epoch 9/30


KeyboardInterrupt: 